# 흉부 X선 영상의 정상·폐렴 분류

Kaggle Chest X-Ray Images (Pneumonia)의 영상 단위 이진 분류 과제다. 일반 CNN과
잔차 CNN을 각각 증강 없이 한 번, 증강을 적용해 한 번 학습한다. 네 조건의 테스트
지표를 비교하고 동일한 X선 영상에서 예측 클래스의 Grad-CAM을 그린다.

**다시 실행하기** · Python 3.11, PyTorch 2.10, Pillow, NumPy, pandas, scikit-learn,
Matplotlib, IPython이 필요하다. Kaggle에서는 데이터셋을 노트북 입력에 추가한다.
로컬에서는 Kaggle에서 내려받은 ZIP 또는 `train/val/test` 폴더를 가진 데이터 경로를
아래 `LOCAL_DATA`에 지정한다. GPU가 있으면 자동으로 사용한다.

- [Kaggle 데이터셋](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)
- [Kermany et al. (2018), 원자료 논문](https://doi.org/10.1016/j.cell.2018.02.010)
- [Selvaraju et al. (2017), Grad-CAM](https://openaccess.thecvf.com/content_iccv_2017/html/Selvaraju_Grad-CAM_Visual_Explanations_ICCV_2017_paper.html)

In [ ]:
import hashlib
import io
import os
import random
import re
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from PIL import Image
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold
from torch import nn
from torch.nn import functional as F

LOCAL_DATA = None  # 예: Path("chest-xray-pneumonia.zip")
SEED = 42
SIZE = 128
BATCH = 64
EPOCHS = 15
THRESHOLD = 0.50
CLASSES = ("NORMAL", "PNEUMONIA")
SPLITS = ("train", "val", "test")
RESULTS = Path(os.environ.get("XRAY_RESULTS_DIR", "xray_results"))
RESULTS.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(6, os.cpu_count() or 1))
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print(f"PyTorch {torch.__version__} | device={DEVICE}")
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name())

## 1. 데이터와 분할

공식 `train`과 `val`을 합쳐 개발 데이터로 사용한다. 공식 `val`의 표본이
적으므로 개발 데이터에서 계층화 그룹 5분할 중 첫 번째 분할을 검증에 사용한다.
파일명에서 추정한 환자 그룹을 기준으로 나누고, 테스트와 동일한 그룹 또는
동일한 JPEG가 있는 개발 데이터는 제거한다. 개발 데이터 내부의 동일 JPEG는
한 장만 남긴다. 공식 `test`는 원래 구성 그대로 최종 평가에 사용한다.
정상 영상은 `IM-번호` 또는 `NORMAL2-IM-번호`를 그룹으로 사용했다. 폐렴 영상은 세균성·바이러스성 표기와 관계없이 같은 `person번호`를 묶는 보수적인 기준을 적용했다. 실제 환자 ID가 제공되지 않아 이 방식은 서로 다른 환자를 같은 그룹으로 묶을 수 있다.

JPEG를 회색조 128×128로 리사이즈한다. 값의 범위를 [0, 1]로 변환한 뒤
학습셋의 평균과 표준편차로 정규화한다.

In [ ]:
IMAGE_SUFFIXES = {".jpeg", ".jpg", ".png"}
REQUIRED_FOLDERS = {(split, label) for split in SPLITS for label in CLASSES}


def dataset_source():
    configured = os.environ.get("XRAY_DATA") or LOCAL_DATA
    if configured:
        path = Path(configured).expanduser()
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    local = [Path("chest_xray"), Path("chest-xray-pneumonia.zip")]
    kaggle = Path("/kaggle/input")
    candidates = local + sorted(kaggle.glob("**/chest_xray"))
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("Kaggle 입력을 추가하거나 LOCAL_DATA에 ZIP 또는 폴더를 지정하세요.")


def archive_records(path):
    """ZIP에 중복 디렉터리 트리가 있으면 정식 chest_xray 트리 하나를 선택한다."""
    with zipfile.ZipFile(path) as archive:
        trees = {}
        for name in archive.namelist():
            parts = name.split("/")
            if len(parts) < 3 or any(part.startswith((".", "__")) for part in parts):
                continue
            split, label, file = parts[-3:]
            if (split, label) not in REQUIRED_FOLDERS or Path(file).suffix.lower() not in IMAGE_SUFFIXES:
                continue
            prefix = tuple(parts[:-3])
            trees.setdefault(prefix, []).append((name, split, CLASSES.index(label)))
    eligible = [
        key for key, rows in trees.items()
        if {(split, CLASSES[y]) for _, split, y in rows} == REQUIRED_FOLDERS
    ]
    if not eligible:
        raise ValueError("ZIP 안에서 train/val/test의 두 클래스 폴더를 찾지 못했습니다.")
    canonical = min(eligible, key=lambda key: (key[-1:] != ("chest_xray",), len(key), key))
    return sorted(trees[canonical])


def folder_records(path):
    candidates = [path, *sorted(path.glob("**/chest_xray"))]
    root = next((
        candidate for candidate in candidates
        if all((candidate / split / label).is_dir() for split, label in REQUIRED_FOLDERS)
    ), None)
    if root is None:
        raise ValueError("폴더 안에서 train/val/test의 두 클래스 폴더를 찾지 못했습니다.")
    rows = [
        (str(file), split, CLASSES.index(label))
        for split, label in sorted(REQUIRED_FOLDERS)
        for file in (root / split / label).iterdir()
        if file.is_file() and not file.name.startswith(".") and file.suffix.lower() in IMAGE_SUFFIXES
    ]
    return sorted(rows)


def group_id(filename, label):
    stem = Path(filename).stem.casefold()
    pneumonia = re.match(r"(person\d+)_(?:bacteria|virus)_", stem)
    normal = re.match(r"((?:normal2-)?im-\d+)(?:-|$)", stem)
    match = pneumonia if label == 1 else normal
    return match.group(1) if match else f"unparsed:{stem}"


def decode_image(raw):
    with Image.open(io.BytesIO(raw)) as image:
        image.draft("L", (SIZE * 2, SIZE * 2))
        resized = image.convert("L").resize((SIZE, SIZE), Image.Resampling.LANCZOS)
        return np.array(resized, dtype=np.uint8)


def load_data(path):
    is_zip = path.is_file()
    rows = archive_records(path) if is_zip else folder_records(path)
    images = np.empty((len(rows), 1, SIZE, SIZE), dtype=np.uint8)
    records = []

    def decode(index_row):
        index, (name, split, label) = index_row
        raw = archive.read(name) if is_zip else Path(name).read_bytes()
        return index, decode_image(raw), {
            "file": Path(name).name, "original_split": split, "label": label,
            "group": group_id(name, label), "sha256": hashlib.sha256(raw).hexdigest(),
        }

    archive = zipfile.ZipFile(path) if is_zip else None
    try:
        with ThreadPoolExecutor(max_workers=6) as pool:
            for index, image, record in pool.map(decode, enumerate(rows)):
                images[index, 0] = image
                records.append(record)
    finally:
        if archive is not None:
            archive.close()
    return pd.DataFrame(records), images


source = dataset_source()
records, images = load_data(source)
original_counts = pd.crosstab(records["original_split"], records["label"].map(dict(enumerate(CLASSES))))
display(original_counts.reindex(SPLITS))

is_test = records["original_split"].eq("test")
test_rows = records[is_test]
development = records[~is_test]
overlap_group = development["group"].isin(test_rows["group"])
overlap_hash = development["sha256"].isin(test_rows["sha256"])
eligible = development[~(overlap_group | overlap_hash)]
duplicate_dev = eligible["sha256"].duplicated()
development = eligible[~duplicate_dev]
print(f"개발 데이터 제외 · 테스트와 그룹 중복 {overlap_group.sum()}장, "
      f"JPEG 중복 {overlap_hash.sum()}장, 두 조건의 합집합 {(overlap_group | overlap_hash).sum()}장")
print(f"개발 데이터 내부의 동일 JPEG 제외 {duplicate_dev.sum()}장")
print(f"파일명에서 그룹 ID를 추정하지 못한 영상 {records['group'].str.startswith('unparsed:').sum()}장")

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_pos, val_pos = next(splitter.split(development, development["label"], development["group"]))
indices = {
    "train": development.index.to_numpy()[train_pos],
    "val": development.index.to_numpy()[val_pos],
    "test": test_rows.index.to_numpy(),
}
records["split"] = "excluded"
for split, index in indices.items():
    records.loc[index, "split"] = split
for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
    for field in ("group", "sha256"):
        assert set(records.loc[indices[left], field]).isdisjoint(records.loc[indices[right], field])
assert all(records.loc[index, "label"].nunique() == 2 for index in indices.values())

split_counts = pd.crosstab(records["split"], records["label"].map(dict(enumerate(CLASSES))))
display(split_counts.reindex(SPLITS))
records.to_csv(RESULTS / "split_manifest.csv", index_label="image_index")
all_images = torch.from_numpy(images).to(DEVICE)
all_labels = torch.tensor(records["label"].to_numpy(), dtype=torch.long, device=DEVICE)
split_indices = {key: torch.as_tensor(index, device=DEVICE) for key, index in indices.items()}
training_pixels = all_images[split_indices["train"]].float().div_(255)
train_std, train_mean = torch.std_mean(training_pixels, correction=0)
del training_pixels
train_std = train_std.clamp_min(1e-6)
train_counts = torch.bincount(all_labels[split_indices["train"]], minlength=2).float()
class_weights = train_counts.sum() / (2 * train_counts)
print(f"학습셋 평균 {train_mean.item():.3f}, 표준편차 {train_std.item():.3f}, "
      f"클래스 가중치 {class_weights.cpu().numpy().round(3)}")

## 2. 일반 CNN과 잔차 CNN

두 모델 모두 무작위 초기화로 학습한다. 합성곱 블록에 BatchNorm과 ReLU를
적용하고, 마지막 특징 맵을 Global Average Pooling한 뒤 두 클래스의 로짓을 계산한다.
일반 CNN은 합성곱을 순차적으로 연결한다. 잔차 CNN은 각 블록 입력을 출력에 더하며,
크기나 채널 수가 바뀌면 1×1 합성곱으로 입력을 맞춘다.

증강 조건에서는 학습 영상에만 회전(±10°), 이동(가로·세로 각각 최대 5%),
확대·축소(스케일 0.92–1.08), 명암 대비(배율 0.85–1.15)를 무작위로 적용한다.
네 조건에 동일한 학습·검증 분할, 가중 교차 엔트로피, AdamW, 코사인 학습률
스케줄과 15에폭 예산을 적용한다. 각 조건에서 검증 ROC-AUC가 가장 높은 에폭의
가중치를 복원한다. 분류 임계값은 사전에 0.50으로 고정한다.

In [ ]:
def seed_everything():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)


def conv_bn(in_channels, out_channels, stride=1, kernel=3):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel, stride, padding=kernel // 2, bias=False),
        nn.BatchNorm2d(out_channels),
    )


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride, residual):
        super().__init__()
        self.conv1 = conv_bn(in_channels, out_channels, stride)
        self.conv2 = conv_bn(out_channels, out_channels)
        self.shortcut = None
        if residual:
            self.shortcut = nn.Identity() if stride == 1 and in_channels == out_channels else conv_bn(
                in_channels, out_channels, stride, kernel=1
            )

    def forward(self, x):
        features = self.conv2(F.relu(self.conv1(x)))
        if self.shortcut is not None:
            features = features + self.shortcut(x)
        return F.relu(features)


class XRayCNN(nn.Module):
    def __init__(self, residual):
        super().__init__()
        self.features = nn.Sequential(
            conv_bn(1, 24, stride=2, kernel=5),
            nn.ReLU(),
            nn.MaxPool2d(2),
            ConvBlock(24, 24, 1, residual),
            ConvBlock(24, 48, 2, residual),
            ConvBlock(48, 96, 2, residual),
            ConvBlock(96, 128, 1, residual),
        )
        self.classifier = nn.Linear(128, len(CLASSES))

    def forward(self, x, return_features=False):
        features = self.features(x)
        logits = self.classifier(features.mean(dim=(-2, -1)))
        if return_features:
            return logits, features
        return logits


def augment_batch(x, generator):
    random_values = torch.rand((5, len(x)), device=x.device, generator=generator) * 2 - 1
    angle, shift_x, shift_y, zoom, contrast = random_values
    angle = angle * np.deg2rad(10)
    scale = 1 / (1 + 0.08 * zoom)
    cosine, sine = scale * angle.cos(), scale * angle.sin()
    theta = torch.stack(
        (cosine, -sine, 0.10 * shift_x, sine, cosine, 0.10 * shift_y), dim=1
    ).reshape(-1, 2, 3)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, padding_mode="border", align_corners=False)
    image_mean = x.mean(dim=(-2, -1), keepdim=True)
    return ((x - image_mean) * (1 + 0.15 * contrast[:, None, None, None]) + image_mean).clamp(0, 1)


def prepare_batch(index, generator=None):
    x = all_images[index].float().div_(255)
    if generator is not None:
        x = augment_batch(x, generator)
    return ((x - train_mean) / train_std).contiguous(memory_format=torch.channels_last)


@torch.inference_mode()
def predict(model, index):
    model.eval()
    scores = [model(prepare_batch(batch)).softmax(dim=1)[:, 1] for batch in index.split(BATCH)]
    return torch.cat(scores).cpu().numpy()


def train_experiment(model_name, augmentation):
    seed_everything()
    model = XRayCNN(residual=model_name == "Residual CNN").to(
        device=DEVICE, memory_format=torch.channels_last
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.002, weight_decay=0.001)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=0.0001)
    use_amp = DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    shuffle_generator = torch.Generator(device=DEVICE).manual_seed(SEED)
    aug_generator = torch.Generator(device=DEVICE).manual_seed(SEED + 1) if augmentation else None
    train_index, val_index = split_indices["train"], split_indices["val"]
    y_val = all_labels[val_index].cpu().numpy()
    history = []
    best_state, best_auc, best_epoch = None, -np.inf, 0
    started = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_index = train_index[torch.randperm(len(train_index), device=DEVICE, generator=shuffle_generator)]
        loss_sum = torch.zeros((), device=DEVICE)
        for batch in epoch_index.split(BATCH):
            x, y = prepare_batch(batch, aug_generator), all_labels[batch]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x)
                loss = F.cross_entropy(logits, y, weight=class_weights)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.detach() * class_weights[y].sum()
        scheduler.step()
        val_auc = roc_auc_score(y_val, predict(model, val_index))
        train_loss = (loss_sum / class_weights[all_labels[train_index]].sum()).item()
        history.append({"epoch": epoch, "train_loss": train_loss, "val_roc_auc": val_auc})
        if val_auc > best_auc:
            best_auc, best_epoch = val_auc, epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        log_line = (
            f"{model_name}, aug={augmentation} | {epoch:02d}/{EPOCHS} | "
            f"loss={train_loss:.4f} | val ROC-AUC={val_auc:.4f}"
        )
        with (RESULTS / "training.log").open("a", encoding="utf-8") as stream:
            stream.write(log_line + "\n")
    model.load_state_dict(best_state)
    seconds = time.perf_counter() - started
    return model, history, best_epoch, best_auc, seconds


EXPERIMENTS = [
    (model_name, augment)
    for model_name in ("Plain CNN", "Residual CNN")
    for augment in (False, True)
]
RUN_NAMES = {
    (name, use_aug): f"{name} / {'aug' if use_aug else 'no aug'}" for name, use_aug in EXPERIMENTS
}
(RESULTS / "training.log").write_text("", encoding="utf-8")
models, histories, validation_results = {}, {}, {}
for key in EXPERIMENTS:
    model, history, best_epoch, best_auc, seconds = train_experiment(*key)
    models[key] = model
    histories[key] = pd.DataFrame(history)
    validation_results[key] = {
        "parameters": sum(p.numel() for p in model.parameters()),
        "best_epoch": best_epoch, "val_roc_auc": best_auc,
    }
    print(f"{RUN_NAMES[key]} | epoch {best_epoch:02d} | val ROC-AUC {best_auc:.4f} | {seconds:.1f}s")
pd.concat([frame.assign(run=RUN_NAMES[key]) for key, frame in histories.items()]).to_csv(
    RESULTS / "learning_curves.csv", index=False
)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), layout="constrained")
colors = dict(zip(EXPERIMENTS, ("#0072b2", "#009e73", "#d55e00", "#a45c93")))
for key, frame in histories.items():
    for axis, field in zip(axes, ("train_loss", "val_roc_auc")):
        axis.plot(frame["epoch"], frame[field], label=RUN_NAMES[key], color=colors[key])
        axis.set(xlabel="Epoch", xticks=range(1, EPOCHS + 1, 2))
        axis.grid(alpha=0.2)
axes[0].set_title("Weighted training loss")
axes[1].set_title("Validation ROC-AUC")
axes[1].legend(fontsize=8, loc="lower right")
display(fig)
plt.close(fig)

## 3. 독립 테스트셋 성능 비교

폐렴을 양성으로 두고 정확도, 균형 정확도, 정밀도, 민감도, 특이도, F1,
ROC-AUC, Average Precision(AP)을 계산한다. 모델별 증강 효과는
`증강 있음 − 증강 없음`, 증강 조건별 모델 차이는 `잔차 CNN − 일반 CNN`으로
표시한다. ROC-AUC 차이의 95% 구간은 같은 테스트 그룹을 두 조건에서 함께
재표집하는 쌍체 부트스트랩(2,000회)으로 구한다. 이 구간은 해당 테스트셋의
표본 변동을 나타내며 재학습에 따른 변동은 포함하지 않는다.

In [ ]:
test_index = split_indices["test"]
y_test = all_labels[test_index].cpu().numpy()
test_scores, test_matrices, metric_rows = {}, {}, []
for key in EXPERIMENTS:
    scores = predict(models[key], test_index)
    pred = (scores >= THRESHOLD).astype(int)
    matrix = confusion_matrix(y_test, pred, labels=[0, 1])
    tn, fp, fn, tp = matrix.ravel()
    test_scores[key], test_matrices[key] = scores, matrix
    metric_rows.append({
        "model": key[0], "augmentation": key[1], **validation_results[key],
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "sensitivity": recall_score(y_test, pred, zero_division=0),
        "specificity": tn / (tn + fp),
        "f1": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, scores),
        "ap": average_precision_score(y_test, scores),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    })
metrics = pd.DataFrame(metric_rows).set_index(["model", "augmentation"])
metrics.to_csv(RESULTS / "test_metrics.csv")
display(metrics.round(4))

test_predictions = records.loc[indices["test"], ["file", "group", "label"]].copy()
for key in EXPERIMENTS:
    test_predictions[RUN_NAMES[key]] = test_scores[key]
test_predictions.to_csv(RESULTS / "test_predictions.csv", index=False)

comparisons = {
    "Plain CNN · aug − no aug": (("Plain CNN", True), ("Plain CNN", False)),
    "Residual CNN · aug − no aug": (("Residual CNN", True), ("Residual CNN", False)),
    "No aug · Residual − Plain": (("Residual CNN", False), ("Plain CNN", False)),
    "Aug · Residual − Plain": (("Residual CNN", True), ("Plain CNN", True)),
}
groups = list(test_predictions.reset_index(drop=True).groupby("group", sort=True).indices.values())
rng = np.random.default_rng(SEED + 2)
bootstrap = []
for _ in range(2_000):
    sampled = np.concatenate([groups[i] for i in rng.integers(0, len(groups), size=len(groups))])
    labels = y_test[sampled]
    if len(np.unique(labels)) < 2:
        continue
    bootstrap.append([roc_auc_score(labels, test_scores[key][sampled]) for key in EXPERIMENTS])
bootstrap = np.asarray(bootstrap)
comparison_rows = []
for label, (left, right) in comparisons.items():
    auc_differences = bootstrap[:, EXPERIMENTS.index(left)] - bootstrap[:, EXPERIMENTS.index(right)]
    lo, hi = np.quantile(auc_differences, [0.025, 0.975])
    comparison_rows.append({
        "comparison": label,
        "delta_accuracy_pp": 100 * (metrics.loc[left, "accuracy"] - metrics.loc[right, "accuracy"]),
        "delta_balanced_accuracy_pp": 100 * (
            metrics.loc[left, "balanced_accuracy"] - metrics.loc[right, "balanced_accuracy"]
        ),
        "delta_f1": metrics.loc[left, "f1"] - metrics.loc[right, "f1"],
        "delta_roc_auc": metrics.loc[left, "roc_auc"] - metrics.loc[right, "roc_auc"],
        "auc_ci_low": lo, "auc_ci_high": hi,
    })
deltas = pd.DataFrame(comparison_rows).set_index("comparison")
deltas.to_csv(RESULTS / "test_comparisons.csv")
display(deltas.round(4))

fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
for key, scores in test_scores.items():
    fpr, tpr, _ = roc_curve(y_test, scores)
    ax.plot(fpr, tpr, color=colors[key], label=f"{RUN_NAMES[key]} ({metrics.loc[key, 'roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], "--", color="0.6", lw=1)
ax.set(xlabel="False positive rate", ylabel="True positive rate", title="ROC curves on the official test set")
ax.grid(alpha=0.2)
ax.legend(loc="lower right", fontsize=8)
display(fig)
plt.close(fig)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.5), layout="constrained")
for axis, key in zip(axes, EXPERIMENTS):
    matrix = test_matrices[key]
    axis.imshow(matrix, cmap="Blues", vmin=0, vmax=len(y_test[y_test == 1]))
    for row, col in np.ndindex(matrix.shape):
        value = matrix[row, col]
        axis.text(col, row, str(value), ha="center", va="center",
                  color="white" if value > 220 else "#16263b", fontsize=13)
    axis.set(
        title=RUN_NAMES[key], xticks=[0, 1], yticks=[0, 1], xticklabels=CLASSES,
        yticklabels=CLASSES, xlabel="Predicted label",
    )
axes[0].set_ylabel("True label")
display(fig)
plt.close(fig)

## 4. Grad-CAM

최종 합성곱 블록의 특징 맵에 대한 예측 클래스 로짓의 기울기를 채널별로
공간 평균한다. 이 가중치로 특징 맵을 합산하고 ReLU를 적용한 뒤 입력 영상 크기로
보간하여 X선 위에 겹쳐 그린다. 각 행에서 네 실험은 동일한 영상을 사용한다.
정상과 폐렴 영상 각 한 장을 시드로 선택하고, 검증 ROC-AUC가 가장 높은 조건의
위양성과 위음성도 각각 한 장 추가한다(해당 오류가 있을 때).
열지도는 각 영상에서 모델의 예측 클래스에 기여한 영역을 나타낸다.
두 모델의 분류기는 GAP 뒤에 선형 층을 사용하므로, 정규화된 양의 Grad-CAM은 분류기 가중치로 계산한 양의 CAM과 같다.

In [ ]:
def grad_cam(model, image_index, target):
    model.eval()
    x = prepare_batch(image_index.reshape(1))
    logits, features = model(x, return_features=True)
    gradients, = torch.autograd.grad(logits[0, target], features)
    weights = gradients.mean(dim=(-2, -1), keepdim=True)
    heatmap = (weights * features).sum(dim=1, keepdim=True).relu()
    heatmap = F.interpolate(heatmap, (SIZE, SIZE), mode="bilinear", align_corners=False)
    heatmap = heatmap / heatmap.amax().clamp_min(1e-8)
    return heatmap[0, 0].detach().cpu().numpy()


reference = max(EXPERIMENTS, key=lambda key: validation_results[key]["val_roc_auc"])
ref_predictions = (test_scores[reference] >= THRESHOLD).astype(int)
sample_rng = np.random.default_rng(SEED + 3)
samples = []
for label in range(len(CLASSES)):
    samples.append((int(sample_rng.choice(np.flatnonzero(y_test == label))), "Class sample"))
for label, description in ((0, "Reference FP"), (1, "Reference FN")):
    selected = [position for position, _ in samples]
    errors = np.setdiff1d(np.flatnonzero((y_test == label) & (ref_predictions != label)), selected)
    if len(errors):
        samples.append((int(sample_rng.choice(errors)), description))

fig, axes = plt.subplots(len(samples), 5, figsize=(13, 2.55 * len(samples)), layout="constrained")
for row, (position, description) in enumerate(samples):
    image_id = int(indices["test"][position])
    original = images[image_id, 0]
    truth = CLASSES[y_test[position]]
    axes[row, 0].imshow(original, cmap="gray", vmin=0, vmax=255)
    filename = records.loc[image_id, "file"]
    axes[row, 0].set_title(f"{description} · {truth}\n{filename}", fontsize=8)
    for col, key in enumerate(EXPERIMENTS, start=1):
        probability = test_scores[key][position]
        target = int(probability >= THRESHOLD)
        cam = grad_cam(models[key], test_index[position], target)
        axis = axes[row, col]
        axis.imshow(original, cmap="gray", vmin=0, vmax=255)
        axis.imshow(cam, cmap="inferno", vmin=0, vmax=1, alpha=0.65 * cam)
        axis.set_title(f"{RUN_NAMES[key]}\n{CLASSES[target]} · P(PNEU)={probability:.2f}", fontsize=8)
    for axis in axes[row]:
        axis.axis("off")
fig.savefig(RESULTS / "grad_cam.png", dpi=150, bbox_inches="tight", facecolor="white")
display(fig)
plt.close(fig)
print(f"오분류 예시 선택 기준: {RUN_NAMES[reference]}")

## 5. 결과 해석

In [ ]:
def interval_statement(low, high):
    if low > 0:
        return "구간 전체가 0보다 높았다"
    if high < 0:
        return "구간 전체가 0보다 낮았다"
    return "구간에 0이 포함됐다"


best = metrics.loc[reference]
lines = [
    f"검증 ROC-AUC로 선택한 조건은 **{RUN_NAMES[reference]}**이다. "
    f"공식 테스트 {len(y_test)}장에서 정확도 **{best['accuracy']:.2%}**, "
    f"ROC-AUC **{best['roc_auc']:.4f}**, F1 **{best['f1']:.4f}**를 기록했다. "
    f"폐렴 {int(best['tp'] + best['fn'])}장 중 {int(best['fn'])}장을 정상으로, "
    f"정상 {int(best['tn'] + best['fp'])}장 중 {int(best['fp'])}장을 폐렴으로 분류했다.",
    "",
]
for label, row in deltas.iterrows():
    lines.append(
        f"- **{label}**에서 ROC-AUC 차이는 {row['delta_roc_auc']:+.4f}"
        f" (95% 구간 {row['auc_ci_low']:+.4f} ~ {row['auc_ci_high']:+.4f})로 "
        f"{interval_statement(row['auc_ci_low'], row['auc_ci_high'])}. "
        f"정확도 차이는 {row['delta_accuracy_pp']:+.2f}%p였다."
    )
lines.extend([
    "",
    "실험은 이 소아 데이터셋의 공식 테스트 분할과 한 개 학습 시드에서 수행했다. "
    "다른 기관이나 연령대에서의 성능은 별도 검증이 필요하다. Grad-CAM은 예측의 "
    "공간적 근거를 보여 주며, 병변 위치와의 일치도는 위치 정답이 없어 평가할 수 없다.",
])
display(Markdown("\n".join(lines)))